# Crop Price Forecasting API for GreenField Farmers
## Task 2: Train and Evaluate Baseline Price Model

**Author:** Mushtaq Ahmad  
**Client:** GreenField Farmers Co-op  
**Objective:** Build a next-week price forecasting baseline using a **Linear Regression** model and benchmark its performance (RMSE, MAE) against a **Naive Persistence Baseline**.

---

### 1. Ingest Libraries & Pipeline Modules

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Ensure project root is available in path
sys.path.append(os.path.abspath('..'))
from src.train_baseline import BaselineModelTrainer

### 2. Loading Cleaned Data & Preparing Target Variable
We create the target variable: `Target_Next_Week_Price` by shifting `Modal_Price_PKR` by -1 week for each crop.

In [ ]:
trainer = BaselineModelTrainer(
    data_path=os.path.join('..', 'data', 'processed', 'crop_prices_cleaned.csv'),
    models_dir=os.path.join('..', 'models')
)
df_model = trainer.load_and_prepare_target()
df_model[['Date', 'Crop', 'Modal_Price_PKR', 'Target_Next_Week_Price']].head(6)

### 3. Chronological Train / Test Split (80% / 20%)
We split temporally to respect the time dimension and prevent future lookahead leakage.

In [ ]:
train_df, test_df = trainer.chronological_train_test_split(df_model, train_ratio=0.80)

plt.figure(figsize=(11, 4))
plt.plot(train_df['Date'], train_df['Modal_Price_PKR'], 'b.', label='Train Data (80%)', alpha=0.5)
plt.plot(test_df['Date'], test_df['Modal_Price_PKR'], 'r.', label='Test Data (20%)', alpha=0.8)
plt.title('Chronological Train/Test Partition')
plt.xlabel('Date')
plt.ylabel('Wholesale Modal Price (PKR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 4. Training Linear Regression & Benchmarking Against Naive Persistence

In [ ]:
metrics_report = trainer.train_and_evaluate()
import json
print(json.dumps(metrics_report, indent=2))

### 5. Visualizing Actual vs. Forecasted Next-Week Prices on Test Set

In [ ]:
model = joblib.load(os.path.join('..', 'models', 'baseline_linear_regression.joblib'))
feature_cols = ['Crop', 'Modal_Price_PKR', 'Min_Price_PKR', 'Max_Price_PKR', 'Market_Arrival_Tonnes', 'Rainfall_mm', 'Temperature_C']
test_df['Predicted_Next_Week'] = model.predict(test_df[feature_cols])

plt.figure(figsize=(12, 6))
for crop in ['Wheat', 'Rice', 'Potato']:
    sub = test_df[test_df['Crop'] == crop]
    plt.plot(sub['Date'], sub['Target_Next_Week_Price'], label=f'{crop} (Actual)', linewidth=2)
    plt.plot(sub['Date'], sub['Predicted_Next_Week'], '--', label=f'{crop} (Baseline Pred)', linewidth=1.8)

plt.title('Next-Week Price Forecasting: Actual vs Linear Regression Baseline (Test Set)', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Price (PKR/Quintal)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()